In [1]:
import pandas as pd
import numpy as np
from LLM import Clasificador
from tqdm import tqdm

c:\Users\chris\Data_science\Proyectos_perso\LLMzCor.github.io\env_PAC\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
file_path = r"C:\Users\chris\Data_science\Proyectos_perso\LLMzCor.github.io\DBs\Clostridium_difficile.xlsx"
sheet = "Test"
usecols= [
    'PMID',
    'Title',
    'Abstract',
    '1) Antimicrobial Resistance stain',
    '2) New treatment',
    '3) Immunization',
    'Human_summary'
]
df = pd.read_excel(
    io=file_path,
    sheet_name=sheet,
    usecols=usecols,
    index_col=0
)

In [3]:
df.head()

,Title,Abstract,1) Antimicrobial Resistance stain,2) New treatment,3) Immunization,Human_summary
PMID,,,,,,
36466927,Neutralizing epitopes on Clostridioides diffic...,Toxin A (TcdA) and toxin B (TcdB) are two key ...,No,Yes,No,Using antibodies (VHHs) AH3 and AA6 are two po...
36439832,Peroxisome proliferator-activated receptor-γ a...,Clostridioides difficile is a major causative ...,No,Yes,No,"Administration of the PPAR-γ agonist, pioglita..."
36439215,The impact of dietary fibers on Clostridioides...,Diets rich in fiber may provide health benefit...,No,Yes,No,Use Inulin or pectin as a dietary-based therap...
36312948,Receptor binding protein of prophage reversibl...,Receptor-binding proteins (RBPs) are located a...,No,Yes,No,The paper studied a protein named PtsHN10M tha...
35042668,The emergence of Clostridioides difficile PCR ...,Background: Several studies have highlighted t...,Yes,No,No,This study focused on analyzing Clostridioides...


In [4]:
clasificador=Clasificador()

In [5]:
def _transformacion_binario_val(col):
    if (col=="Yes") | (col==1):
        return 1
    elif (col=="No") | (col==0):
        return 0
def _transformacion_binario_df(df):
    df[['1) Antimicrobial Resistance stain',
        '2) New treatment','3) Immunization']] = df[['1) Antimicrobial Resistance stain',
                                                     '2) New treatment',
                                                     '3) Immunization']].applymap(_transformacion_binario_val)
    return df



def ask_llm(df_,partition=0):
    df=df_.copy()
    df=_transformacion_binario_df(df)
    df["ai_label"]=np.nan
    df["ai_summary"]=np.nan
    if partition==0:
        for pmid in df.index:
            try:
                response = clasificador.clasificacion(df.loc[pmid,"Abstract"])
                df.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
            except Exception as e:
                print(f"Se corto el proceso del LLM por este motivo : {e} en el id {pmid}")
                return df.loc[:pmid].iloc[:-1]
        return df
    else:
        print("aca")
        df_ptit=df.iloc[:partition,:]
        for pmid in df_ptit.index:
            try:
                response = clasificador.clasificacion(df_ptit.loc[pmid,"Abstract"])
                df_ptit.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
            except Exception as e:
                print(f"Se corto el proceso del LLM por este motivo : {e} en el id {pmid}")
                return df_ptit.loc[:pmid].iloc[:-1]
        return df_ptit

def _evaluate(row):
    values = row[["1) Antimicrobial Resistance stain","2) New treatment","3) Immunization"]].values
    ai_opcion=int(row["ai_label"])-1
    print(f"values={values}")
    print(f"ai_opcion{ai_opcion}")
    if (values.sum()==0) & (ai_opcion==3):
        return 1
    elif (values.sum()==0) & (ai_opcion<3):
        return 0
    elif (values.sum()>0) & (ai_opcion==3):
        return 0
    elif values[ai_opcion]>0:

        return 1
    else:
        return 0

def evaluacion_score(df,partition=0):
    if partition ==0:
        n=df.shape[0]
        scores = df.apply(_evaluate,axis=1)
        final_score=sum(scores)/n
    else:
        df_ptit=df.iloc[:10,:]
        n=df_ptit.shape[0]
        scores=df_ptit.apply(_evaluate,axis=1)
        final_score=sum(scores)/n
    return final_score

# 1ero : ask_llm(df)

Llamas a la funcion "df_new = ask_llm(df_old)"

Esto te va a a dar un nuevo df_new

In [6]:
df_new=ask_llm(df)

C:\Users\chris\AppData\Local\Temp\ipykernel_30952\3837796252.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)


Se corto el proceso del LLM por este motivo : 402 Client Error: Payment Required for url: https://api-inference.huggingface.co/models/mistralai/Mixtral-8x7B-Instruct-v0.1 (Request ID: Root=1-681ca897-42689e5521cf10015938105f;c54f64e3-59ee-4940-ad3e-755aed931503)

You have exceeded your monthly included credits for Inference Providers. Subscribe to PRO to get 20x more monthly included credits. en el id 36466927


Ahora podras ver tu nuevo df_new con ai_label y ai_summary

In [36]:
df_new.head()

,Title,Abstract,1) Antimicrobial Resistance stain,2) New treatment,3) Immunization,Human_summary,ai_label,ai_summary
PMID,,,,,,,,
36466927,Neutralizing epitopes on Clostridioides diffic...,Toxin A (TcdA) and toxin B (TcdB) are two key ...,0,1,0,Using antibodies (VHHs) AH3 and AA6 are two po...,3,"""The paper discusses the use of single-domain..."
36439832,Peroxisome proliferator-activated receptor-γ a...,Clostridioides difficile is a major causative ...,0,1,0,"Administration of the PPAR-γ agonist, pioglita...",2,'The paper discusses a new potential treatmen...
36439215,The impact of dietary fibers on Clostridioides...,Diets rich in fiber may provide health benefit...,0,1,0,Use Inulin or pectin as a dietary-based therap...,2,"""The paper discusses the use of pectin as a p..."
36312948,Receptor binding protein of prophage reversibl...,Receptor-binding proteins (RBPs) are located a...,0,1,0,The paper studied a protein named PtsHN10M tha...,4,"""The paper does not discuss Multiresistance b..."
35042668,The emergence of Clostridioides difficile PCR ...,Background: Several studies have highlighted t...,1,0,0,This study focused on analyzing Clostridioides...,1,"""The paper discusses the incidence of Clostri..."


# EVALUATION SCORE

Ahora podras obtener el score de tu data data frame con evaluacion_score(df_new)

In [35]:
evaluacion_score(df_new)

values=[0 1 0]
ai_opcion2
values=[0 1 0]
ai_opcion1
values=[0 1 0]
ai_opcion1
values=[0 1 0]
ai_opcion3
values=[1 0 0]
ai_opcion0
values=[1 0 0]
ai_opcion0
values=[1 0 0]
ai_opcion3
values=[1 0 0]
ai_opcion3
values=[1 0 0]
ai_opcion3
values=[0 1 0]
ai_opcion1
values=[0 0 1]
ai_opcion3
values=[0 0 0]
ai_opcion2
values=[1 0 0]
ai_opcion3
values=[1 0 0]
ai_opcion0
values=[1 0 0]
ai_opcion0
values=[0 1 0]
ai_opcion1
values=[0 0 1]
ai_opcion2
values=[1 0 0]
ai_opcion3
values=[0 1 0]
ai_opcion1
values=[1 0 0]
ai_opcion3
values=[0 1 0]
ai_opcion1
values=[1 0 0]
ai_opcion0
values=[0 1 0]
ai_opcion1
values=[0 1 0]
ai_opcion3
values=[0 0 1]
ai_opcion2
values=[0 1 0]
ai_opcion1
values=[0 1 0]
ai_opcion3
values=[0 1 0]
ai_opcion1
values=[0 0 1]
ai_opcion2
values=[1 0 0]
ai_opcion0
values=[0 1 0]
ai_opcion1
values=[1 0 0]
ai_opcion3
values=[0 1 0]
ai_opcion1
values=[0 1 0]
ai_opcion3
values=[0 1 0]
ai_opcion1
values=[1 1 0]
ai_opcion1
values=[0 1 0]
ai_opcion1
values=[0 1 0]
ai_opcion1
values=[0 1 

0.62